# What actually matters: auto-routing + distillation

Two features a real user cares about:

1. **Auto-routing** — one line loads a pre-trained router that picks the right model per prompt (quality + cost), using learned clusters and per-model error profiles. No engine needed for this part; runs locally in pure Python.
2. **Distillation** — submit a job, train a small student model from teacher traces, and land a distilled model behind a routing alias. Requires the engine running (`make start-full`).

Everything below makes real calls. No mocks.

In [1]:
!pip install sentence-transformers

In [2]:
import os, sys, time, json

SRC = os.path.abspath(os.path.join(os.getcwd(), ".."))
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import lunar_router as lr
print(f"lunar-router {lr.__version__}")

# Engine/API probe — used later for the distillation section
import urllib.request
def _probe(url):
    try:
        with urllib.request.urlopen(url, timeout=1) as r:
            return r.status == 200
    except Exception:
        return False

ENGINE_UP = _probe("http://localhost:8080/health")
API_UP    = _probe("http://localhost:8000/health")
print(f"engine (:8080):  {'up' if ENGINE_UP else 'DOWN'}")
print(f"rest api (:8000): {'up' if API_UP else 'DOWN'}")

lunar-router 0.1.0
engine (:8080):  up
rest api (:8000): up


## Part 1 — Auto-routing

One call downloads pre-trained weights (first run, ~100MB from HuggingFace) and returns a `UniRouteRouter`. Each `.route(prompt)` returns a `RoutingDecision` with:
- `selected_model` — the model chosen for this prompt
- `cluster_id` — which semantic cluster the prompt landed in (out of ~100)
- `expected_error` — predicted error rate for the selected model on this cluster
- `all_scores` — score per model (lower = better, cost-weighted)
- `reasoning` — human-readable explanation

In [3]:
# Default backend is the Go engine — faster, production path.
# If the `lunar-engine` binary isn't on PATH in this environment, fall back to
# the Python backend so the demo still runs. In a real install you'd:
#     pip install lunar-router[engine]   # ships the binary
#   OR build from source: cd go && make build
t = time.time()
try:
    router = lr.load_router(cost_weight=0.5, verbose=False)   # engine="go" (default)
    backend = "go"
except FileNotFoundError as e:
    print(f"go binary not found — falling back to python backend\n  ({str(e).splitlines()[0]})")
    router = lr.load_router(cost_weight=0.5, verbose=False, engine="python")
    backend = "python"

print(f"router loaded in {time.time()-t:.1f}s  backend={backend}")


go binary not found — falling back to python backend
  (lunar-engine binary not found. Options:)


/home/ubuntu/ml-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10874.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


router loaded in 24.7s  backend=python


/home/ubuntu/lunar-router/lunar_router/core/embeddings.py:260: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self.model.get_sentence_embedding_dimension()


In [ ]:
# Route a handful of prompts with different difficulty profiles
PROMPTS = [
    ("factual",  "What is the capital of France?"),
    ("math",     "Prove that the square root of 2 is irrational."),
    ("code",     "Write a Python function that reverses a linked list in place."),
    ("creative", "Write a haiku about a lonely lighthouse."),
    ("reasoning","A bat and a ball cost $1.10 together. The bat costs $1 more than the ball. How much does the ball cost?"),
    ("long",     "Summarize the plot of War and Peace in three paragraphs."),
]

header = f"{'kind':<10} {'model':<32} {'cluster':>8} {'err':>7}  prompt"
print(header)
print("-" * len(header))
for kind, p in PROMPTS:
    d = router.route(p)
    print(f"{kind:<10} {d.selected_model:<32} {d.cluster_id:>8} {d.expected_error:>7.3f}  {p[:60]}...")

In [ ]:
# Inspect one decision in full — the reasoning + top-5 candidate scores
d = router.route(PROMPTS[2][1])  # the code prompt
print("prompt:", PROMPTS[2][1])
print("\n--- reasoning ---")
print(d.reasoning)
print("\n--- top 5 candidate models (lower score = better) ---")
for model, score in sorted(d.all_scores.items(), key=lambda x: x[1])[:5]:
    print(f"  {model:<40} {score:.4f}")

### Does the learned router beat naive strategies?

Compare three strategies over a batch of prompts:
- **learned** — our router's pick
- **always-smartest** — pin everything to the best available model (quality ceiling, highest cost)
- **always-cheapest** — pin everything to the weakest available model (cost floor, quality risk)

We use the same per-model error profiles the router learned from, so the comparison is apples-to-apples on *expected* error, not real calls.

In [ ]:
import numpy as np

# A slightly bigger eval batch
batch = [
    "What year did World War II end?",
    "Explain the difference between a stack and a queue.",
    "What is the derivative of sin(x) * x^2?",
    "Write a short poem about autumn.",
    "Translate 'good morning' to French, Spanish, and Japanese.",
    "Why is the sky blue?",
    "List 5 prime numbers greater than 100.",
    "What is the time complexity of quicksort?",
    "Describe the lifecycle of a butterfly.",
    "What's 17 * 23?",
    "Explain recursion to a 10-year-old.",
    "Summarize the causes of the French Revolution.",
    "Write a SQL query to find the top 3 customers by revenue.",
    "What is entropy in information theory?",
    "Generate a JSON schema for a user profile with name, age, email.",
]

# Pick always-smartest / always-cheapest from the registry the router was loaded with.
reg       = router.registry
all_models = reg.get_model_ids()
profiles   = {m: reg.get(m) for m in all_models}
costs      = {m: profiles[m].cost_per_1k_tokens for m in all_models}
cheapest       = min(all_models, key=lambda m: costs[m])
most_expensive = max(all_models, key=lambda m: costs[m])
print(f"cheapest model in registry:     {cheapest:<30} ${costs[cheapest]:.5f}/1k")
print(f"most expensive in registry:     {most_expensive:<30} ${costs[most_expensive]:.5f}/1k\n")

errors    = {"learned": [], "always-smartest": [], "always-cheapest": []}
costs_arr = {"learned": [], "always-smartest": [], "always-cheapest": []}

for p in batch:
    d = router.route(p)
    errors["learned"].append(d.expected_error)
    costs_arr["learned"].append(costs[d.selected_model])

    # Baselines read the same per-cluster error profile, but for the fixed model
    errors["always-smartest"].append(profiles[most_expensive].get_cluster_error(d.cluster_id))
    errors["always-cheapest"].append(profiles[cheapest].get_cluster_error(d.cluster_id))
    costs_arr["always-smartest"].append(costs[most_expensive])
    costs_arr["always-cheapest"].append(costs[cheapest])

print(f"{'strategy':<20} {'avg err':>10} {'avg cost/1k':>14}")
print("-" * 46)
for strat in ["always-cheapest", "learned", "always-smartest"]:
    print(f"{strat:<20} {np.mean(errors[strat]):>10.4f} {np.mean(costs_arr[strat]):>14.6f}")


The learned router should land between the two extremes: close to always-smartest on error, closer to always-cheapest on price. That's the cost/quality tradeoff — tuneable via `cost_weight` (λ) at load time.

---

## Part 2 — Distillation

This is the wedge. A user's trace data (captured via `completion()`) becomes a dataset, a teacher model generates high-quality labels, and a small student model is fine-tuned on the result. The distilled student is served behind a routing alias — the user's app keeps calling `model="smart"` and the traffic transparently moves to the cheaper model.

`Distiller` is the REST client. It requires the engine + API server. If `make start-full` isn't running, the cells below will print instructions and skip the live calls.

In [ ]:
# The SDK client always instantiates cleanly — it's just a configured httpx session.
d = lr.Distiller(base_url="http://localhost:8000")
print(f"client: {d}  base_url={d.base_url}")
print(f"live methods: {[m for m in dir(d) if not m.startswith('_') and callable(getattr(d, m))]}")

In [ ]:
# Discover what teacher/student models the engine currently supports.
# These lists come from the engine's registry, not a hardcoded SDK constant.
if not API_UP:
    print("REST API not reachable. Start it with `make start-full`, then rerun this cell.")
else:
    teachers = d.teacher_models()
    students = d.student_models()
    print(f"available teachers ({len(teachers)}):")
    for t in teachers[:6]:
        print(f"  - {t['id']:<45} [{t.get('provider','?')}]")
    if len(teachers) > 6:
        print(f"  ... +{len(teachers)-6} more")
    print(f"\navailable students ({len(students)}):")
    for s in students[:6]:
        print(f"  - {s['id']:<25} family={s.get('family','?'):<10} params={s.get('params','?')}")
    if len(students) > 6:
        print(f"  ... +{len(students)-6} more")

In [ ]:
# Dry-run cost + balance estimate for a small distillation job.
# This is what a user would call BEFORE committing to a run.
if not API_UP:
    print("skipped: API not up")
else:
    est = d.estimate(
        student_model="llama-3.2-1b",
        num_prompts=20,     # small demo
        n_samples=2,        # BOND: 2 teacher samples per prompt
    )
    for k, v in est.items():
        print(f"  {k:<20} {v}")

In [ ]:
# The full job submission — shown but NOT executed unless SUBMIT=1.
# A real run takes minutes-to-hours and uses GPU; opt in explicitly.
SUBMIT = os.environ.get("SUBMIT", "0") == "1"

job_config = dict(
    name="llama-3.2-1b ← gpt-4o-mini demo",
    student_model="llama-3.2-1b",
    teacher_model="openai/gpt-4o-mini",
    num_prompts=20,
    n_samples=2,
    training_steps=30,
    bond_beta=0.5,    # BOND loss weights
    bond_gamma=0.1,
    temperature=0.8,
    export_gguf=False,  # skip GGUF export for a quick demo
)
print("job config:")
print(json.dumps(job_config, indent=2))

if SUBMIT and API_UP:
    job = d.create(**job_config)
    print(f"\nsubmitted: job_id={job['id']}  status={job['status']}")
    print("poll with: d.wait(job['id'])  or  stream with: d.stream_progress(job['id'])")
else:
    print("\n(not submitted — set SUBMIT=1 env var to actually kick off the job)")

## How the two features fit together

```
user app → lr.completion(model="smart", ...) → auto-router picks a provider
                                              → traces captured (prompt, response, cost)
                                              → traces become a dataset
                                              → Distiller trains a student from teacher labels
                                              → alias "smart" gets swapped to the cheaper student
                                              → user app never changed a line of code
```

The auto-router gives you day-1 value (cost/quality tradeoff without training). Distillation is the compounding value — over weeks, your cheap-cluster traffic migrates to distilled models you own.